In [1]:
import sys
%pip install "urllib3<2"
print(sys.executable)

Note: you may need to restart the kernel to use updated packages.
/Users/andresho/Desktop/Research/Cooperative_MARL/venv/bin/python


In [2]:
import numpy as np
import time
from dataclasses import dataclass
import tensorflow as tf
from game_environments.ToyGame import ToyGame
from misc import *
from networks import *
from rollout import *
from updates import *
from agents import build_agents

tf.random.set_seed(0)
np.random.seed(0)

@dataclass
class Config:
    N: int = 2            # number of agents
    n_actions: int = 2    # binary actions (0/1)
    n_states: int = 2     # s1 (XNOR), s2 (XOR)
    ep_len: int = 100     # timesteps per episode (paper uses 100)
    gamma: float = 0.99
    p_no_adv: float = 0.4 # p(no adversary); else one agent (uniform) is adversary

cfg = Config()
agents = build_agents(cfg, hidden=128)
belief_step = make_belief_step(agents)
print("TF", tf.__version__, "| Config:", cfg)

TF 2.20.0 | Config: Config(N=2, n_actions=2, n_states=2, ep_len=100, gamma=0.99, p_no_adv=0.4)


# Training

In [3]:
def train(env_cls, cfg, agents, belief_step, B=1000, n_iters=1000, team_interval=1, log_every=50):
    hist = {"reward_adv": [], "reward_clean": [], "belief_acc": []}
    t0 = time.time()
    for it in range(1, n_iters + 1):
        traj = run_episodes_batched(env_cls, cfg, B, agents, belief_step)
        G, adv = compute_advantages_batched(traj, agents, cfg)
        update_adversary(traj, adv, agents, cfg)
        if it % team_interval == 0:
            update_critic(traj, G, agents, cfg)
            update_belief(traj, agents, cfg)
            update_policy(traj, adv, agents, cfg)
        if it % log_every == 0:
            ev = run_episodes_batched(env_cls, cfg, B, agents, belief_step)
            ep_reward = ev['reward'].mean(axis=1)
            has_adv = ev['theta'][:, 0, :].sum(axis=1) > 0
            ra = ep_reward[has_adv].mean()  if has_adv.any()   else float('nan')
            rc = ep_reward[~has_adv].mean() if (~has_adv).any() else float('nan')
            acc = ((ev['belief'] > 0.5).astype(int) == ev['theta']).mean()
            hist["reward_adv"].append(ra); hist["reward_clean"].append(rc)
            hist["belief_acc"].append(acc)
            print(f"iter {it:5d} | reward(adv) {ra:.3f} | reward(clean) {rc:.3f} "
                  f"| belief acc {acc:.3f} | {time.time()-t0:.0f}s")
    return hist

In [ ]:
hist = train(
    ToyGame, cfg, agents, belief_step, 
    B=1000, n_iters=1000, log_every=100,
    team_interval=2
)

iter   100 | reward(adv) 0.552 | reward(clean) 0.504 | belief acc 0.692 | 78s


In [5]:
import time

traj = run_episodes_batched(ToyGame, cfg, 1000, agents, belief_step)  # warmup

def time_block(fn, name, reps=5):
    t0 = time.time()
    for _ in range(reps):
        out = fn()
    dt = (time.time() - t0) / reps
    print(f"{name:25s} {dt*1000:7.1f} ms")
    return out

print("=== timing one iteration's pieces (B=1000) ===")
traj = time_block(lambda: run_episodes_batched(ToyGame, cfg, 1000, agents, belief_step),
                  "run_episodes_batched")
G, adv = compute_advantages_batched(traj, agents, cfg)
time_block(lambda: compute_advantages_batched(traj, agents, cfg), "compute_advantages")
time_block(lambda: update_adversary(traj, adv, agents, cfg), "update_adversary")
time_block(lambda: update_critic(traj, G, agents, cfg), "update_critic")
time_block(lambda: update_belief(traj, agents, cfg), "update_belief")
time_block(lambda: update_policy(traj, adv, agents, cfg), "update_policy")

=== timing one iteration's pieces (B=1000) ===
run_episodes_batched        495.4 ms
compute_advantages           21.2 ms
update_adversary            155.7 ms
update_critic                84.2 ms
update_belief               375.6 ms
update_policy               182.1 ms


0.07957751303911209

In [ ]:
# --- Inspect the action-selection block on a tiny batch ---
from misc import sample_types_batched, belief_features_batched, sample_batched

B = 4                                   # tiny batch so output is readable
env = ToyGame(cfg, B)
theta = sample_types_batched(cfg, B)    # who's corrupt this episode
h = tf.zeros((B, 128))
prev_actions = np.zeros((B, cfg.N), dtype=int)

# one timestep of setup
obs_all = env._obs()                    # (B, N, n_states)
state_oh = obs_all[:, 0, :]             # (B, n_states)
f = belief_features_batched(state_oh, prev_actions, cfg)
b_tf, h = belief_step(tf.constant(f), h)
b_t = b_tf.numpy()                      # (B, N) belief

print("theta (who's corrupt):\n", theta)
print("\nbelief b_t [P(agent0 corrupt), P(agent1 corrupt)]:\n", b_t.round(3))

# ---- the block you asked about ----
print("\n--- cooperative actions (per agent, using egocentric belief) ---")
a_coop_list = []
for i in range(cfg.N):
    b_ego = np.roll(b_t, -i, axis=1)                    # roll so agent i's slot is first
    logits = agents.policy([state_oh, b_ego]).numpy()   # (B, n_actions)
    a_i = sample_batched(logits)                        # (B,)
    a_coop_list.append(a_i)
    print(f"agent {i}: b_ego=\n{b_ego.round(3)}\n         logits=\n{logits.round(3)}\n         sampled action={a_i}")
a_coop = np.stack(a_coop_list, axis=1)                  # (B, N)

print("\n--- adversary actions ---")
a_logits = agents.adversary(obs_all.reshape(B*cfg.N, cfg.n_states)).numpy().reshape(B, cfg.N, -1)
a_adv = np.stack([sample_batched(a_logits[:, i, :]) for i in range(cfg.N)], axis=1)
print("a_adv (what each agent WOULD do if corrupted):\n", a_adv)

print("\n--- apply corruption ---")
actions = np.where(theta == 1, a_adv, a_coop)
print("a_coop (cooperative choices):\n", a_coop)
print("a_adv  (sabotage choices)   :\n", a_adv)
print("theta  (corrupt mask)       :\n", theta)
print("actions = where(theta==1, a_adv, a_coop):\n", actions)